In [1]:
from ast_skills.data_gen import retriever_batch_join
import pathlib

ds = retriever_batch_join.build_summary_retriever_data_models(
    pathlib.Path("done_summary/"),
    batch_results_dir=pathlib.Path("batch_results_summary/"),
    skill_md_records_path=pathlib.Path("batch_summary_inputs/skill_md_records.jsonl")
)

2026-04-14 18:11:34.995 | INFO     | ast_skills.data_gen.retriever_batch_join:_iter_jsonl_files:58 - directory=PosixPath('done_summary'), len(paths)=52
2026-04-14 18:11:34.997 | INFO     | ast_skills.data_gen.retriever_batch_join:_iter_jsonl_files_recursive:65 - directory=PosixPath('batch_results_summary'), len(paths)=52
2026-04-14 18:11:39.595 | INFO     | ast_skills.data_gen.retriever_batch_join:_load_skill_md_records_by_custom_id:174 - path=PosixPath('batch_summary_inputs/skill_md_records.jsonl'), len(rows)=33472, len(by_id)=33472
2026-04-14 18:11:39.751 | WARNING  | ast_skills.data_gen.dataset:first_valid_summary_extraction_from_output_rows:185 - SkillMdSummaryExtraction parsing failed for custom_id='sm-10516'
2026-04-14 18:11:39.782 | WARNING  | ast_skills.data_gen.dataset:first_valid_summary_extraction_from_output_rows:185 - SkillMdSummaryExtraction parsing failed for custom_id='sm-10673'
2026-04-14 18:11:40.084 | WARNING  | ast_skills.data_gen.dataset:first_valid_summary_extract

In [ ]:
# import json
# import pathlib

# artifacts_dir = pathlib.Path("artifacts")
# artifacts_dir.mkdir(parents=True, exist_ok=True)
# with open(artifacts_dir / "summary_retriever_models.jsonl", "w") as f:
#     for obj in ds:
#         json.dump(obj.__dict__, f)
#         f.write("\n")

In [1]:
from ast_skills.data_gen.datamodels import SummaryRetrieverDataModel
import json


summary_path = "artifacts/summary_retriever_models.jsonl"
summary_data = []
with open(summary_path, "r") as f:
    for line in f:
        obj = json.loads(line)
        item = SummaryRetrieverDataModel(
            custom_id=obj.get("custom_id", ""),
            markdown_content=obj.get("markdown_content", ""),
            seed_questions=obj.get("seed_questions", []),
            summary=obj.get("summary", ""),
            name=obj.get("name", ""),
            description=obj.get("description", ""),
            metadata=obj.get("metadata", {}),
        )
        summary_data.append(item)

In [4]:
# for idx,sd in enumerate(summary_data):
#     if not sd.name:
#         print(idx)

In [10]:
def read_jsonl(path):
    """Reads a JSONL file and returns a list of dicts."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

rows = read_jsonl("skill_md_records_1.jsonl")

In [18]:
import yaml
import datetime
import re

# Fenced frontmatter with both delimiters: --- ... ---
_FRONTMATTER_BOTH_RE = re.compile(r"\A---\s*\r?\n(.*?)\r?\n---\s*(?:\r?\n|$)", re.DOTALL)
# Fenced frontmatter with only the opening ---  (no closing delimiter found)
_FRONTMATTER_OPEN_RE = re.compile(r"\A---\s*\r?\n(.*)", re.DOTALL)
# Direct key extraction: value runs until the next bare "key:" line or end of string.
_NAME_RE = re.compile(r"^name:\s*(.+)$", re.MULTILINE)
_DESCRIPTION_RE = re.compile(r"^description:\s*(.*?)(?=\n\w[\w -]*:|\Z)", re.MULTILINE | re.DOTALL)
# Lone surrogates cannot be encoded to UTF-8 for JSONL output.
_SURROGATE_RE = re.compile(r"[\ud800-\udfff]")

def _metadata_value_to_str(value: object) -> str:
    """Coerce a YAML value to a string for dict[str, str] storage."""
    if value is None:
        return ""
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, datetime.datetime):
        return value.isoformat()
    if isinstance(value, datetime.date):
        return value.isoformat()
    if isinstance(value, (dict, list)):
        return json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            default=str,
        )
    return str(value)


def _extract_fenced_block(text: str) -> tuple[str, str]:
    """
    Return (yaml_block, body) for a fenced --- frontmatter.

    Handles three cases:
      - Both opening and closing ``---`` present → yaml_block is the content
        between them; body is everything after the closing delimiter.
      - Only the opening ``---`` present → yaml_block is everything after it;
        body is empty.
      - No ``---`` at all → both are empty strings.
    """
    match = _FRONTMATTER_BOTH_RE.match(text)
    if match:
        return match.group(1), text[match.end() :]

    match = _FRONTMATTER_OPEN_RE.match(text)
    if match:
        return match.group(1), ""

    return "", ""


def _parse_yaml_block(yaml_text: str) -> dict[str, str]:
    """
    Parse a YAML string into a dict[str, str].

    Returns an empty dict on parse failure or if the result is not a mapping.
    """
    try:
        loaded = yaml.safe_load(yaml_text)
    except yaml.YAMLError:
        return {}

    if not isinstance(loaded, dict):
        return {}

    return {str(key): _metadata_value_to_str(val) for key, val in loaded.items()}


def _extract_name(text: str) -> str:
    """Return the value of the first ``name:`` line, or empty string."""
    match = _NAME_RE.search(text)
    return match.group(1).strip() if match else ""


def _extract_description(text: str) -> str:
    """
    Return the value of the first ``description:`` field.

    The value continues until the next bare ``key:`` line or end of string,
    so multi-line descriptions are captured correctly.
    """
    match = _DESCRIPTION_RE.search(text)
    return match.group(1).strip() if match else ""


def parse_skill_md_frontmatter(raw_text: str) -> tuple[dict[str, str], str]:
    """
    Split SKILL.md into frontmatter metadata and markdown body.

    Strategy:
      1. ``name`` and ``description`` are always extracted directly by regex so
         they are found even when no ``---`` delimiters exist.
      2. All other metadata keys come from the fenced ``---`` / ``---`` block
         when present (missing closing delimiter is tolerated).
      3. The body is whatever follows the closing ``---``; empty if the closing
         delimiter was absent.

    All values are stored as strings (nested YAML structures are JSON-encoded).
    ``name`` and ``description`` are always present (empty string if missing).
    """
    text = raw_text.lstrip("\ufeff")

    yaml_block, body = _extract_fenced_block(text)
    meta = _parse_yaml_block(yaml_block) if yaml_block else {}

    # Direct regex extraction wins for name / description because it works
    # even without --- delimiters and is more tolerant of malformed YAML.
    name = _extract_name(text)
    description = _extract_description(text)

    if name:
        meta["name"] = name
    if description:
        meta["description"] = description

    meta.setdefault("name", "")
    meta.setdefault("description", "")
    return meta, body

In [22]:
parse_skill_md_frontmatter(rows[5]['content'])

({'name': '', 'description': ''}, '')

In [31]:
bad_names = 0
for idx,sd in enumerate(summary_data):
    if not sd.name:
        d,m = parse_skill_md_frontmatter(sd.markdown_content)
        if d['name'] == '':
            print(d, sd.markdown_content[:100])
            bad_names+=1

{'name': '', 'description': ''} # 🤝 AI Partnership & JV Deal Finder — Find Hidden Revenue Partners & Close Deals in 10 Minutes

---

{'name': '', 'description': ''} # 💼 AI Personal Branding Kit — LinkedIn + Bio + Content Strategy in One Run

**Slug:** `ai-personal-
{'name': '', 'description': ''} # 🧠 AI Personal Finance Advisor — Build Your Complete Wealth Plan & Find $1,000+ in Hidden Savings i
{'name': '', 'description': ''} # 🏠 AI Real Estate Goldmine Finder — Spot Undervalued Properties & Hot Markets Before Everyone Else

{'name': '', 'description': ''} # 🏠 AI Real Estate Video Marketing Engine — List, Market & Sell Any Property Faster With AI-Produced
{'name': '', 'description': ''} # 🏆 AI Recruiting Intelligence Engine: Find, Score and Contact Top Candidates Before Any Other Recru
{'name': '', 'description': ''} # 👥 AI Recruitment & Talent Scout Engine

**Slug:** `ai-recruitment-talent-scout`  
**Category:** HR
{'name': '', 'description': ''} # 🧲 AI Reddit Lead Mining Machine — F

In [32]:
bad_names

3778